# Figure 1 — Stochasticity Analysis

**Goal:** Replicate Figure 1 of the paper on toy datasets.

We compute the cosine similarity between the optimal velocity field $\hat{u}^\star$ and the conditional target $u^{\text{cond}} = x_1 - x_0$.

- **1a.** Cosine similarity histograms for various $t$ on 2D datasets.
- **1c.** Dimension dependence: percentage of samples with $\cos > 0.9$ vs $t$ for varying $d$.
- **Comparison with Biroli et al.** Empirical collapse time vs theoretical $t_{\text{collapse}} \sim \mathcal{O}(\log(n)/d)$.

In [ ]:
# Colab setup
import os, sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/YOUR_REPO/ClosedFlowMatching.git 2>/dev/null || true
    os.chdir('ClosedFlowMatching')
    !pip install -q -r requirements.txt
    sys.path.insert(0, '.')
else:
    sys.path.insert(0, '..')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

from src.data.toy import ToyDataset
from src.flow_matching.closed_form import optimal_velocity, cosine_sim_u_star_vs_ucond

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False
})
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1.1 Cosine similarity histograms (Figure 1a)

For 256 pairs $(x_0, x_1) \sim p_0 \times \hat{p}_{\text{data}}$, we compute at each $t$:

$$\cos\bigl(\hat{u}^\star((1-t)x_0 + t\,x_1,\, t),\; x_1 - x_0\bigr)$$

On 2D data, we expect the collapse to happen at intermediate-to-large $t$.

In [ ]:
n_pairs = 256
t_display = [0.01, 0.1, 0.2, 0.4, 0.6, 0.8, 0.95]
datasets_2d = ['moons', 'rings', 'gaussian_mixture']

fig, axes = plt.subplots(len(datasets_2d), len(t_display), figsize=(18, 3 * len(datasets_2d)))

for row, ds_name in enumerate(datasets_2d):
    ds = ToyDataset(ds_name, n_samples=5000)
    data = ds.data.to(device)

    x1 = ds.sample(n_pairs).to(device)
    x0 = torch.randn_like(x1)

    cos_dict = cosine_sim_u_star_vs_ucond(x0, x1, data, t_display)

    for col, t_val in enumerate(t_display):
        ax = axes[row, col]
        vals = cos_dict[t_val].cpu().numpy()
        ax.hist(vals, bins=40, range=(-1, 1), color='steelblue', edgecolor='white', linewidth=0.3)
        ax.set_xlim(-1, 1)
        if row == 0:
            ax.set_title(f't = {t_val:.2f}')
        if col == 0:
            ax.set_ylabel(ds_name)
        ax.set_yticks([])

fig.suptitle('Cosine similarity between $\\hat{u}^\\star$ and $u^{\\mathrm{cond}}$', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig1a_cosine_histograms.pdf', bbox_inches='tight')
plt.show()

## 1.2 Dimension dependence (Figure 1c)

We use `gaussian_mixture_nd` at dimensions $d \in \{2, 10, 50, 100, 500, 1000\}$.

For each $d$ and $t$, we plot the percentage of pairs with $\cos > 0.9$.

The paper shows that as $d$ grows, collapse to a single conditional velocity happens at earlier times.
We average over 8 random seeds to reduce noise (low-dimensional curves are noisy with 256 pairs).

**Note on low-$d$ non-monotonicity:** In $d=2$, it is expected that the curve may be non-monotone: at very early $t$, the softmax weights are nearly uniform and the average velocity may accidentally align with $x_1 - x_0$ for a few samples; then the alignment worsens as the weights redistribute before finally converging at large $t$. This effect vanishes in higher dimensions due to concentration of measure.

In [ ]:
dims = [2, 10, 50, 100, 500, 1000]
t_grid = torch.linspace(0.01, 0.99, 80)
n_pairs = 256
n_samples = 2000
n_seeds = 8  # average over multiple batches to reduce noise
threshold = 0.9

fig, ax = plt.subplots(figsize=(8, 5))
colors = cm.viridis(np.linspace(0.15, 0.95, len(dims)))

for i, d in enumerate(dims):
    ds = ToyDataset('gaussian_mixture_nd', n_samples=n_samples, dim=d)
    data = ds.data.to(device)

    pct_all_seeds = np.zeros((n_seeds, len(t_grid)))
    for seed in range(n_seeds):
        torch.manual_seed(seed)
        x1 = ds.sample(n_pairs).to(device)
        x0 = torch.randn(n_pairs, d, device=device)

        for j, t_val in enumerate(t_grid):
            t = torch.full((n_pairs, 1), t_val.item(), device=device)
            x_t = (1.0 - t) * x0 + t * x1
            u_star = optimal_velocity(x_t, data, t)
            u_cond = x1 - x0
            cos = torch.nn.functional.cosine_similarity(u_star, u_cond, dim=-1)
            pct_all_seeds[seed, j] = (cos > threshold).float().mean().item() * 100

    pct_mean = pct_all_seeds.mean(axis=0)
    ax.plot(t_grid.numpy(), pct_mean, label=f'd = {d}', color=colors[i], linewidth=2)

ax.set_xlabel('t')
ax.set_ylabel(f'% samples with cos > {threshold}')
ax.set_title('Dimension dependence of stochasticity collapse')
ax.legend()
ax.set_ylim(-5, 105)
plt.tight_layout()
plt.savefig('fig1c_dimension_dependence.pdf', bbox_inches='tight')
plt.show()

## 1.3 Empirical collapse time vs Biroli et al. theory

Biroli et al. predict $t_{\text{collapse}} \sim \mathcal{O}(\log(n)/d)$ for Gaussian mixtures.

We define the empirical collapse time as the smallest $t$ such that $\geq 90\%$ of samples have cosine similarity $> 0.9$, and compare it to the theoretical scaling.

In [ ]:
dims_for_collapse = [2, 5, 10, 20, 50, 100, 200, 500, 1000]
n_samples_list = [500, 2000]

# Fine grid near 0 where collapse happens in high dims, plus coarser grid for large t
t_fine = torch.cat([
    torch.linspace(0.001, 0.05, 50),
    torch.linspace(0.06, 0.99, 100)
])

pct_threshold = 90
cos_threshold = 0.9
n_pairs_collapse = 512
n_seeds_collapse = 4  # average over seeds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, n_samp in enumerate(n_samples_list):
    collapse_times = []
    for d in dims_for_collapse:
        ds = ToyDataset('gaussian_mixture_nd', n_samples=n_samp, dim=d)
        data = ds.data.to(device)

        t_collapse_seeds = []
        for seed in range(n_seeds_collapse):
            torch.manual_seed(seed + 100)
            x1 = ds.sample(n_pairs_collapse).to(device)
            x0 = torch.randn(n_pairs_collapse, d, device=device)

            t_c = 1.0
            for t_val in t_fine:
                t = torch.full((n_pairs_collapse, 1), t_val.item(), device=device)
                x_t = (1.0 - t) * x0 + t * x1
                u_star = optimal_velocity(x_t, data, t)
                u_cond = x1 - x0
                cos = torch.nn.functional.cosine_similarity(u_star, u_cond, dim=-1)
                pct = (cos > cos_threshold).float().mean().item() * 100
                if pct >= pct_threshold:
                    t_c = t_val.item()
                    break
            t_collapse_seeds.append(t_c)
        collapse_times.append(np.mean(t_collapse_seeds))

    dims_arr = np.array(dims_for_collapse, dtype=float)
    theory = np.log(n_samp) / dims_arr
    # Scale theory to match empirical at d=2 (first point)
    theory_scaled = theory / theory[0] * collapse_times[0]

    ax = axes[ax_idx]
    ax.plot(dims_arr, collapse_times, 'o-', color='steelblue', linewidth=2, label='Empirical')
    ax.plot(dims_arr, theory_scaled, '--', color='tomato', linewidth=2, label=r'$\propto \log(n)/d$')
    ax.set_xlabel('Dimension d')
    ax.set_ylabel('Collapse time $t_C$')
    ax.set_title(f'n = {n_samp}')
    ax.legend()
    ax.set_xscale('log')
    ax.set_ylim(bottom=-0.02)

fig.suptitle('Empirical collapse time vs. Biroli et al. prediction', fontsize=13)
plt.tight_layout()
plt.savefig('fig1_collapse_time_vs_theory.pdf', bbox_inches='tight')
plt.show()

## 1.4 Collapse time vs number of samples n

Fix $d$ at low values ($d = 5$ and $d = 10$) and vary $n$ to check the $\log(n)$ factor.

**Important:** $d$ must be small enough that the collapse time is measurable (not clamped at the grid start). For $d = 50$, $\log(n)/d$ is already $< 0.2$ for all reasonable $n$, making the variation invisible.

In [ ]:
d_values_for_n = [5, 10]
n_values = [50, 100, 200, 500, 1000, 2000, 5000, 10000]

t_fine_n = torch.cat([
    torch.linspace(0.001, 0.1, 100),
    torch.linspace(0.11, 0.99, 90)
])

fig, axes = plt.subplots(1, len(d_values_for_n), figsize=(7 * len(d_values_for_n), 5))
if len(d_values_for_n) == 1:
    axes = [axes]

for ax_idx, d_fixed in enumerate(d_values_for_n):
    collapse_vs_n = []
    for n_samp in n_values:
        ds = ToyDataset('gaussian_mixture_nd', n_samples=n_samp, dim=d_fixed)
        data = ds.data.to(device)

        t_collapse_seeds = []
        for seed in range(n_seeds_collapse):
            torch.manual_seed(seed + 200)
            x1 = ds.sample(n_pairs_collapse).to(device)
            x0 = torch.randn(n_pairs_collapse, d_fixed, device=device)

            t_c = 1.0
            for t_val in t_fine_n:
                t = torch.full((n_pairs_collapse, 1), t_val.item(), device=device)
                x_t = (1.0 - t) * x0 + t * x1
                u_star = optimal_velocity(x_t, data, t)
                u_cond = x1 - x0
                cos = torch.nn.functional.cosine_similarity(u_star, u_cond, dim=-1)
                if (cos > cos_threshold).float().mean().item() * 100 >= pct_threshold:
                    t_c = t_val.item()
                    break
            t_collapse_seeds.append(t_c)
        collapse_vs_n.append(np.mean(t_collapse_seeds))

    n_arr = np.array(n_values, dtype=float)
    theory_n = np.log(n_arr) / d_fixed
    theory_n_scaled = theory_n / theory_n[0] * collapse_vs_n[0]

    ax = axes[ax_idx]
    ax.plot(n_arr, collapse_vs_n, 'o-', color='steelblue', linewidth=2, label='Empirical')
    ax.plot(n_arr, theory_n_scaled, '--', color='tomato', linewidth=2, label=r'$\propto \log(n)/d$')
    ax.set_xlabel('n (dataset size)')
    ax.set_ylabel('Collapse time $t_C$')
    ax.set_title(f'Collapse time vs n (d = {d_fixed})')
    ax.set_xscale('log')
    ax.legend()

plt.tight_layout()
plt.savefig('fig1_collapse_vs_n.pdf', bbox_inches='tight')
plt.show()